# Store Sales: compare forecasting models

One notebook that runs the chosen models and shows one results table, timings and a chart.
All code lives in the `store_sales/` package next to this notebook.

| Model | What it is | Runs on |
|---|---|---|
| Baseline | Average of the same weekday over the last 4 weeks | driver |
| LightGBM | One model for all series, features built with pandas | driver |
| LightGBM (Spark features) | Same model, features built with Spark | features: workers; model: driver |
| Prophet | One model per series | driver's cores |
| Prophet (Spark) | Same model, one Spark task per series | workers |
| ARIMA / ARIMAX | One AutoARIMA per series; ARIMAX adds promotions and holidays | driver's cores, own process on AIDP |
| ARIMA (Spark) / ARIMAX (Spark) | Same models, one Spark task per series | workers |

Every model is scored on the same 3 back-to-back 16-day windows with RMSLE (lower is better).

## Settings

In [ ]:
PROJECT_DIR = "."  # AIDP: "/Workspace/forecasting/time_series_data" (the folder holding store_sales/ and data/)
DATA_DIR = f"{PROJECT_DIR}/data"

SAMPLE = 50  # score this many fixed random series; 0 = all 1,782 (LightGBM always trains on all of them)

MODELS = [
    "Baseline",
    "LightGBM",
    "LightGBM (Spark features)",
    "Prophet (Spark)",
    "ARIMA (Spark)",
    "ARIMAX (Spark)",
    # Driver-only versions (same results, slow on a small AIDP driver):
    # "Prophet", "ARIMA", "ARIMAX",
]

ARIMA_IN_OWN_PROCESS = True  # driver-only ARIMA/ARIMAX on AIDP: statsforecast can't be imported in the notebook session

In [ ]:
import os
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, os.path.abspath(PROJECT_DIR))
from store_sales import arima_model, lgbm_model, prophet_model
from store_sales.data import backtest, baseline, load_train, only, sample_series

train = load_train(DATA_DIR)
scored = only(train, sample_series(train, n=SAMPLE)) if SAMPLE else train
print(f"scoring {scored.groupby(['store_nbr', 'family']).ngroups} series, driver CPU cores: {os.cpu_count()}")

spark = globals().get("spark")  # AIDP notebooks provide `spark`; on a laptop it doesn't exist yet
if any("Spark" in m for m in MODELS):
    from store_sales.spark_setup import get_spark
    spark = get_spark(spark)  # configures AIDP's session, or starts a local one on a laptop
    print("Spark parallelism (worker cores):", spark.sparkContext.defaultParallelism)

## Run

In [ ]:
scores, seconds = {}, {}


def score(name, forecaster, extra_seconds=0.0, note=""):
    t = time.time()
    scores[name] = backtest(scored, forecaster)
    seconds[name] = time.time() - t + extra_seconds
    print(f"{name:28} mean RMSLE {scores[name]['mean']:.4f}   {seconds[name]:6.0f}s  {note}")


if "Baseline" in MODELS:
    score("Baseline", baseline)

if "LightGBM" in MODELS:
    t = time.time()
    features = lgbm_model.add_features(train, DATA_DIR)
    built = time.time() - t
    score("LightGBM", lgbm_model.make_forecaster(features, train), built, f"(incl. {built:.0f}s pandas features)")

if "LightGBM (Spark features)" in MODELS:
    t = time.time()
    one_day = lgbm_model.add_features(train[train.date == train.date.max()], DATA_DIR)  # column order + categories
    spark_features = lgbm_model.add_features_spark(spark, train, DATA_DIR, like=one_day)
    built = time.time() - t
    score("LightGBM (Spark features)", lgbm_model.make_forecaster(spark_features, train), built,
          f"(incl. {built:.0f}s Spark features)")

prophets = prophet_model.make_forecasters(DATA_DIR, spark)
for name in ["Prophet", "Prophet (Spark)"]:
    if name in MODELS:
        score(name, prophets[name])

arima_forecasters = arima_model.make_forecasters(DATA_DIR, spark)
for name in ["ARIMA (Spark)", "ARIMAX (Spark)"]:
    if name in MODELS:
        score(name, arima_forecasters[name])

arimas = [m for m in ["ARIMA", "ARIMAX"] if m in MODELS]
if arimas and ARIMA_IN_OWN_PROCESS:
    for name, r in arima_model.run_in_subprocess(DATA_DIR, arimas, sample=SAMPLE).items():
        scores[name], seconds[name] = pd.Series(r["scores"]), r["seconds"]
elif arimas:
    for name in arimas:
        score(name, arima_forecasters[name])

## Results

In [ ]:
table = pd.DataFrame(scores).T
table = table[[c for c in table.columns if c != "mean"] + ["mean"]].sort_values("mean")
table["seconds"] = pd.Series(seconds).round(0)
print(f"{'all 1,782' if not SAMPLE else SAMPLE} series, RMSLE per backtest window (lower is better)")
table.round(4)

In [ ]:
ax = table["mean"].sort_values(ascending=False).plot.barh(figsize=(8, 0.5 * len(table) + 1), color="#4C78A8")
ax.set_xlabel("mean RMSLE over 3 windows (lower is better)")
ax.set_title(f"Model comparison, {'all 1,782' if not SAMPLE else SAMPLE} series")
for i, v in enumerate(table["mean"].sort_values(ascending=False)):
    ax.text(v, i, f" {v:.4f}", va="center")
plt.tight_layout()
plt.show()

In [ ]:
# Keep a copy of this run's results next to the notebook
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)
out = f"{PROJECT_DIR}/results/compare_{'all' if not SAMPLE else SAMPLE}_{time.strftime('%Y%m%d_%H%M')}.csv"
table.to_csv(out)
print("saved", out)